# G10 — Notebook 01 : Exploration du dataset Allociné (D05)

**Groupe 10** | CamemBERT-base × Allociné | Problématique P02  
**Objectif** : Explorer le dataset avant l'optimisation

---

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
print('Imports OK')

## 1. Chargement du dataset

In [ ]:
from src.data_loader import load_raw_dataset, dataset_stats

dataset = load_raw_dataset()
print(dataset)

In [ ]:
# Statistiques descriptives
dataset_stats(dataset)

## 2. Analyse de la longueur des critiques

In [ ]:
# Longueur des critiques (en mots)
train_data = dataset['train']
lengths = [len(ex['review'].split()) for ex in train_data.select(range(2000))]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(lengths, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(256, color='red', lw=2, ls='--', label='MAX_SEQ_LENGTH=256')
axes[0].set_xlabel('Longueur (mots)')
axes[0].set_ylabel('Fréquence')
axes[0].set_title('Distribution des longueurs (train, 2k exemples)')
axes[0].legend()

axes[1].boxplot([lengths], labels=['Train'])
axes[1].set_ylabel('Longueur (mots)')
axes[1].set_title('Boîte à moustaches des longueurs')

plt.tight_layout()
plt.savefig('../results/figures/length_distribution.png', bbox_inches='tight')
plt.show()

trunc_pct = sum(1 for l in lengths if l > 256) / len(lengths) * 100
print(f'Critiques tronquées (>256 mots) : {trunc_pct:.1f}%')

## 3. Exemple de critiques

In [ ]:
# Afficher quelques exemples
label_names = {0: '😠 Négatif', 1: '😊 Positif'}

for label in [0, 1]:
    examples = [ex for ex in dataset['train'] if ex['label'] == label][:2]
    print(f'\n{'='*60}')
    print(f'  {label_names[label]}')
    print('='*60)
    for ex in examples:
        print(f'\n"{ex["review"][:300]}..."')

## 4. Sous-échantillonnage pour CPU

In [ ]:
from src.data_loader import balanced_subsample
from src.config import N_TRAIN_PER_CLASS, N_VAL_PER_CLASS

train_subset = balanced_subsample(dataset['train'], n_per_class=N_TRAIN_PER_CLASS)
val_subset = balanced_subsample(dataset['validation'], n_per_class=N_VAL_PER_CLASS)

print(f'Train subset : {len(train_subset)} exemples')
print(f'Val subset   : {len(val_subset)} exemples')

# Vérification de l'équilibre
train_labels = Counter([ex['label'] for ex in train_subset])
print(f'Distribution train : {dict(train_labels)}')

## 5. Analyse du tokenizer CamemBERT

In [ ]:
from src.model_setup import load_tokenizer

tokenizer = load_tokenizer()

# Longueur en tokens (≠ mots)
sample_texts = [ex['review'] for ex in train_subset[:500]]
token_lengths = [len(tokenizer.encode(t, truncation=False)) for t in sample_texts]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(token_lengths, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
ax.axvline(256, color='red', lw=2, ls='--', label='Troncature à 256')
ax.axvline(512, color='darkred', lw=2, ls=':', label='Max CamemBERT (512)')
ax.set_xlabel('Longueur (tokens SentencePiece)')
ax.set_ylabel('Fréquence')
ax.set_title('Distribution des longueurs après tokenisation CamemBERT')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/token_lengths.png', bbox_inches='tight')
plt.show()

print(f'Longueur moyenne : {np.mean(token_lengths):.0f} tokens')
print(f'Médiane : {np.median(token_lengths):.0f} tokens')
print(f'% tronqués à 256 : {sum(1 for l in token_lengths if l > 256)/len(token_lengths)*100:.1f}%')

## 6. Aperçu du modèle CamemBERT

In [ ]:
from src.model_setup import load_model, model_summary

# Charger avec le dropout par défaut (0.1)
model, device = load_model(dropout=0.1)
model_summary(model)
print(f'Device : {device}')